In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D13 — Unemployment Rates by Country of Birth
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter
from openpyxl import load_workbook

import hashlib
import json
import platform
import re
import sys

import pandas as pd


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D13"

DOCUMENT_NAME = (
    "Eurostat — Unemployment rates by country of birth"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".xlsx"

INPUT_REPRESENTATION = "Original XLSX workbook"

DIRECT_DOCUMENT_INGESTION = True


# ------------------------------------------------------------
# Workbook structure
# ------------------------------------------------------------

SUMMARY_SHEET = "Summary"

EXPECTED_SHEETS = [
    "Summary"
] + [
    f"Sheet {number}"
    for number in range(1, 16)
]


# ------------------------------------------------------------
# Fixed Stage 1 extraction scope
# ------------------------------------------------------------

SELECTED_SHEETS = [
    "Sheet 1",
    "Sheet 2",
    "Sheet 3",
    "Sheet 4",
    "Sheet 5"
]


SELECTED_GEOGRAPHIES = [
    "European Union - 27 countries (from 2020)",
    "Belgium",
    "Germany",
    "Spain",
    "Portugal"
]


SELECTED_YEARS = [
    2020,
    2022,
    2024
]

EXPECTED_RECORD_COUNT = (
    len(SELECTED_SHEETS)
    * len(SELECTED_GEOGRAPHIES)
    * len(SELECTED_YEARS)
)


EXPECTED_YEAR_COUNTS = {
    str(year):
        len(SELECTED_SHEETS)
        * len(SELECTED_GEOGRAPHIES)

    for year in SELECTED_YEARS
}


# ------------------------------------------------------------
# Exact Stage 1 schema / constants
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]


EXPECTED_CATEGORY = (
    "Labour market time series"
)


EXPECTED_TOPIC = (
    "Unemployment rate by country of birth"
)


EXPECTED_UNIT = "percent"


EXPECTED_CATEGORY_COUNTS = {
    EXPECTED_CATEGORY:
        EXPECTED_RECORD_COUNT
}


ALLOWED_CATEGORIES = {
    EXPECTED_CATEGORY
}


STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D13_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_representation.json"
)

SELECTION_SCOPE_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_selection_scope.csv"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_technical_diagnostics.json"
)

SCOPE_CHECK_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_scope_check.csv"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D13_branch_A_experiment_summary.json"
)


print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Representation:", INPUT_REPRESENTATION)
print("Selected sheets:", SELECTED_SHEETS)
print("Selected geographies:", SELECTED_GEOGRAPHIES)
print("Selected years:", SELECTED_YEARS)

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D13 XLSX workbook."
)


uploaded = files.upload()


xlsx_paths = [
    Path(filename)
    for filename in uploaded
    if filename.lower().endswith(".xlsx")
]


if len(xlsx_paths) != 1:

    raise ValueError(
        "Upload exactly one XLSX workbook."
    )


SOURCE_PATH = xlsx_paths[0]


# ------------------------------------------------------------
# SHA-256 utility
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


def clean_text(value):

    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)


FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


# ------------------------------------------------------------
# Diagnostic workbook loading only
# ------------------------------------------------------------

workbook = load_workbook(
    SOURCE_PATH,
    data_only=True,
    read_only=False
)


sheet_names = workbook.sheetnames


print(
    "Source file:",
    SOURCE_PATH.name
)

print(
    "Source SHA-256:",
    SOURCE_SHA256
)

print(
    "File size:",
    FILE_SIZE_BYTES
)

print(
    "Worksheet count:",
    len(sheet_names)
)

print(
    "Worksheets:",
    sheet_names
)

In [ ]:
# ============================================================
# 3. Extraction-scope diagnostics
# ============================================================

worksheet_count_valid = (
    len(sheet_names) == 16
)


worksheet_names_valid = (
    sheet_names == EXPECTED_SHEETS
)


summary_sheet_present = (
    SUMMARY_SHEET
    in sheet_names
)


selected_sheets_present = all(
    sheet in sheet_names
    for sheet in SELECTED_SHEETS
)


# ------------------------------------------------------------
# Summary metadata
# ------------------------------------------------------------

summary_worksheet = workbook[
    SUMMARY_SHEET
]


summary_text = " ".join(
    clean_text(
        cell.value
    )

    for row
    in summary_worksheet.iter_rows()

    for cell
    in row

    if cell.value is not None
)


summary_metadata_valid = all([
    "lfsa_urgacob"
    in summary_text,

    "Unemployment rates by country of birth"
    in summary_text,

    "Annual"
    in summary_text,

    "Percentage"
    in summary_text,

    "From 15 to 74 years"
    in summary_text
])


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def find_year_columns(worksheet):

    year_columns = {}

    for column_index in range(
        1,
        worksheet.max_column + 1
    ):

        value = worksheet.cell(
            row=11,
            column=column_index
        ).value


        if (
            isinstance(value, int)
            and not isinstance(value, bool)
        ):

            year_columns[
                value
            ] = column_index


        elif (
            isinstance(value, str)
            and re.fullmatch(
                r"20\d{2}",
                value.strip()
            )
        ):

            year_columns[
                int(
                    value.strip()
                )
            ] = column_index


    return year_columns


def find_geography_rows(worksheet):

    geography_rows = {}

    for row_index in range(
        13,
        worksheet.max_row + 1
    ):

        label = clean_text(
            worksheet.cell(
                row=row_index,
                column=1
            ).value
        )


        if label:

            geography_rows[
                label
            ] = row_index


    return geography_rows


# ------------------------------------------------------------
# Derive structural scope
# ------------------------------------------------------------

scope_rows = []


for sheet_name in SELECTED_SHEETS:

    worksheet = workbook[
        sheet_name
    ]


    year_columns = find_year_columns(
        worksheet
    )


    geography_rows = find_geography_rows(
        worksheet
    )


    for geography in SELECTED_GEOGRAPHIES:

        if geography not in geography_rows:

            raise AssertionError(
                f"{geography!r} was not found in {sheet_name}."
            )


        row_index = geography_rows[
            geography
        ]


        for year in SELECTED_YEARS:

            if year not in year_columns:

                raise AssertionError(
                    f"Year {year} was not found in {sheet_name}."
                )


            value_column = year_columns[
                year
            ]


            flag_column = (
                value_column + 1
            )


            value_cell = worksheet.cell(
                row=row_index,
                column=value_column
            )


            flag_cell = worksheet.cell(
                row=row_index,
                column=flag_column
            )


            scope_rows.append(
                {
                    "Sheet":
                        sheet_name,

                    "Geography":
                        geography,

                    "Year":
                        year,

                    "Expected Value Cell":
                        value_cell.coordinate,

                    "Expected Flag Cell":
                        flag_cell.coordinate,

                    "Expected Source Location":
                        (
                            f"{sheet_name}, "
                            f"cell {value_cell.coordinate}"
                        )
                }
            )


selection_scope_df = pd.DataFrame(
    scope_rows
)


scope_record_count_valid = (
    len(selection_scope_df)
    == EXPECTED_RECORD_COUNT
)


expected_source_locations = (
    selection_scope_df[
        "Expected Source Location"
    ].tolist()
)


expected_source_location_set = set(
    expected_source_locations
)


expected_source_locations_unique = (
    len(
        expected_source_location_set
    )
    == EXPECTED_RECORD_COUNT
)


scope_valid = all([
    scope_record_count_valid,
    expected_source_locations_unique
])


# ------------------------------------------------------------
# Overall source integrity
# ------------------------------------------------------------

INPUT_INTEGRITY_PASSED = all([
    FILE_NON_EMPTY,
    worksheet_count_valid,
    worksheet_names_valid,
    summary_sheet_present,
    selected_sheets_present,
    summary_metadata_valid,
    scope_valid
])


selection_scope_df.to_csv(
    SELECTION_SCOPE_PATH,
    index=False,
    encoding="utf-8-sig"
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "source_format":
        SOURCE_FORMAT,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "expected_worksheet_count":
        16,

    "observed_worksheet_count":
        len(
            sheet_names
        ),

    "worksheet_count_valid":
        worksheet_count_valid,

    "expected_worksheet_names":
        EXPECTED_SHEETS,

    "observed_worksheet_names":
        sheet_names,

    "worksheet_names_valid":
        worksheet_names_valid,

    "summary_sheet_present":
        summary_sheet_present,

    "selected_sheets":
        SELECTED_SHEETS,

    "selected_sheets_present":
        selected_sheets_present,

    "selected_geographies":
        SELECTED_GEOGRAPHIES,

    "selected_years":
        SELECTED_YEARS,

    "summary_metadata_valid":
        summary_metadata_valid,

    "scope_record_count_valid":
        scope_record_count_valid,

    "expected_source_locations_unique":
        expected_source_locations_unique,

    "scope_valid":
        scope_valid,

    "machine_readable":
        True,

    "ocr_required":
        False,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Worksheet count valid:",
    worksheet_count_valid
)

print(
    "Worksheet names valid:",
    worksheet_names_valid
)

print(
    "Summary sheet present:",
    summary_sheet_present
)

print(
    "Selected sheets present:",
    selected_sheets_present
)

print(
    "Summary metadata valid:",
    summary_metadata_valid
)

print(
    "Scope records:",
    len(
        selection_scope_df
    )
)

print(
    "Expected source locations unique:",
    expected_source_locations_unique
)

print(
    "Input integrity passed:",
    INPUT_INTEGRITY_PASSED
)


display(
    selection_scope_df.head(20)
)


if not INPUT_INTEGRITY_PASSED:

    raise AssertionError(
        "D13 source workbook integrity validation failed."
    )

In [ ]:
# ============================================================
# 4. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "source_representation":
        (
            "Multi-sheet Eurostat XLSX workbook with worksheet-level "
            "dimensions, geography rows, annual value columns and "
            "adjacent statistical-flag cells"
        ),

    "complete_original_document_supplied":
        True,

    "direct_document_ingestion":
        True,

    "diagnostic_workbook_loading_applied":
        True,

    "diagnostic_worksheet_inspection_applied":
        True,

    "diagnostic_cell_coordinate_inspection_applied":
        True,

    "diagnostic_scope_derivation_applied":
        True,

    "diagnostically_inspected_cells_used_as_model_input":
        False,

    "derived_representation_used_as_model_input":
        False,

    "xlsx_to_text_conversion_applied":
        False,

    "xlsx_to_csv_conversion_applied":
        False,

    "worksheet_extraction_applied_to_model_input":
        False,

    "worksheet_filtering_applied_to_model_input":
        False,

    "row_filtering_applied_to_model_input":
        False,

    "column_filtering_applied_to_model_input":
        False,

    "column_restructuring_applied":
        False,

    "row_reordering_applied":
        False,

    "aggregation_applied":
        False,

    "interpolation_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "value_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "model_input_description":
        (
            "The complete original D13 XLSX workbook is supplied "
            "directly to the LLM. OpenPyXL workbook loading, worksheet "
            "inspection and cell-coordinate inspection are used only "
            "for source-integrity and extraction-scope diagnostics. "
            "No derived worksheet representation or scope table is "
            "supplied to the model."
        )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 5. Extraction prompt
# ============================================================

sheet_lines = "\n".join(
    f"- {sheet_name}"
    for sheet_name
    in SELECTED_SHEETS
)


geography_lines = "\n".join(
    f"- {geography}"
    for geography
    in SELECTED_GEOGRAPHIES
)


year_lines = "\n".join(
    f"- {year}"
    for year
    in SELECTED_YEARS
)


BRANCH_A_PROMPT = f"""You are an information extraction assistant.

Extract the predefined unemployment-rate observations represented in
the attached original XLSX workbook:

"Unemployment rates by country of birth"

Treat the attached original workbook as the only source of information.


Selected worksheets:

{sheet_lines}


Selected geographies:

{geography_lines}


Selected reporting years:

{year_lines}


For every represented combination of selected worksheet, selected
geography and selected reporting year, return one record.

For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location


Use these fixed semantic values:

Category:
Labour market time series

Topic:
Unemployment rate by country of birth

Unit:
percent


Worksheet-level dimensions:

For each selected worksheet, read the represented worksheet-level
dimensions directly from the workbook:

- Sex
- Age Class
- Country/Region of Birth

Do not infer these dimensions from the worksheet number or from another
worksheet.


Description:

Construct Description in exactly this order:

Geography: <geography>; Sex: <sex>; Age class: <age class>;
Country/region of birth: <birth category>

Use the dimension wording represented in the corresponding worksheet.

When the statistical-flag cell directly adjacent to the selected value
cell is non-empty, append:

; Statistical flag: <flag>

Do not append a Statistical flag segment when the adjacent flag cell
is blank.

Preserve the represented statistical flag exactly.
Do not infer, rewrite, expand or interpret statistical flags.


Value:

- Extract the numerical unemployment-rate value represented for the
  selected geography and selected reporting year.
- Use the value cell, not the adjacent statistical-flag cell.
- Return the value as a JSON number.
- Do not calculate, aggregate, interpolate, round, convert or correct
  the represented value.


Reporting Period:

- Use the selected reporting year as a four-digit string.
- Example:
  "2024"


Source Location:

- Identify the exact workbook cell containing the numerical value.
- Use this form:
  "Sheet N, cell A1"
- Use the value cell, not the adjacent statistical-flag cell.
- Determine the cell directly from the workbook.


Extraction rules:

- Use only the selected worksheets.
- Use only the selected geographies.
- Use only the selected reporting years.
- Return one observation for every represented combination within this
  predefined scope.
- Do not omit a required combination.
- Do not return observations outside the predefined scope.
- Read Sex, Age Class and Country/Region of Birth directly from each
  selected worksheet.
- Preserve exact geography and dimension labels.
- Preserve an adjacent statistical flag only when explicitly represented.
- Do not calculate missing observations.
- Do not aggregate values.
- Do not interpolate values.
- Do not convert percentages into another scale.
- Do not infer missing statistical flags.
- Do not use external knowledge.
- Do not follow external links.
- Do not use observations from unselected worksheets.
- Do not duplicate records.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.


Expected JSON structure:

{{
  "document_id": "D13",
  "branch": "A",
  "records": [
    {{
      "Category": "Labour market time series",
      "Topic": "Unemployment rate by country of birth",
      "Description": null,
      "Value": null,
      "Unit": "percent",
      "Reporting Period": null,
      "Source Location": null
    }}
  ]
}}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print()

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open a new independent ChatGPT conversation.

Upload:

1. the complete original D13 XLSX workbook;
2. `D13_branch_A_prompt.txt`.

Submission of the prompt once.

Save the complete untouched model response as:

`D13_branch_A_raw_response.txt`


In [ ]:
# ============================================================
# 6. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D13_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".txt")
]


if len(txt_paths) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = (
    txt_paths[0]
)


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


# ------------------------------------------------------------
# Preserve untouched response BEFORE parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


# ------------------------------------------------------------
# Non-crashing parser
# ------------------------------------------------------------

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )


# ------------------------------------------------------------
# Standard wrapper checks
# ------------------------------------------------------------

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)


records_evaluable = all([
    valid_json,
    top_level_object_valid,
    records_present,
    records_is_list
])


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None


# ------------------------------------------------------------
# Parsed extraction only if evaluable
# ------------------------------------------------------------

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True


    parsed_extraction_sha256 = sha256_file(
        PARSED_EXTRACTION_PATH
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df
    )

In [ ]:
# ============================================================
# 7. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# A. Exact schema
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if observed_fields != EXPECTED_FIELDS:

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Field names or field order differ",

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field
                            for field
                            in EXPECTED_FIELDS
                            if field not in record
                        ],

                    "extra_fields":
                        [
                            field
                            for field
                            in observed_fields
                            if field not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue[
            "record_index"
        ]
        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues
        == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# B. Field types
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        # String or null
        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        # Value = number or null
        value = record.get(
            "Value"
        )


        if (
            value is not None
            and (
                isinstance(
                    value,
                    bool
                )
                or not isinstance(
                    value,
                    (int, float)
                )
            )
        ):

            field_type_issues.append(
                {
                    "record_index":
                        record_index,

                    "field":
                        "Value",

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "number or null"
                }
            )


        # Mandatory content
        for field in MANDATORY_CONTENT_FIELDS:

            field_value = record.get(
                field
            )


            if (
                field_value is None
                or (
                    isinstance(
                        field_value,
                        str
                    )
                    and not field_value.strip()
                )
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue[
            "record_index"
        ]
        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues
        == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count
        == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# C. Count/category/year diagnostics
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
        )
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    observed_year_counts = dict(
        Counter(
            str(
                record.get(
                    "Reporting Period"
                )
            ).strip()

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
        )
    )


    year_counts_valid = (
        observed_year_counts
        == EXPECTED_YEAR_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    category_counts_valid = None

    categories_valid = None

    observed_year_counts = None

    year_counts_valid = None


# ------------------------------------------------------------
# D. Constant fields
# ------------------------------------------------------------

if records_evaluable:

    category_constant_valid = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Category"
        )
        == EXPECTED_CATEGORY

        for record
        in extracted_records
    )


    topic_constant_valid = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Topic"
        )
        == EXPECTED_TOPIC

        for record
        in extracted_records
    )


    unit_constant_valid = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Unit"
        )
        == EXPECTED_UNIT

        for record
        in extracted_records
    )


    constant_fields_valid = all([
        category_constant_valid,
        topic_constant_valid,
        unit_constant_valid
    ])


else:

    category_constant_valid = None

    topic_constant_valid = None

    unit_constant_valid = None

    constant_fields_valid = None


# ------------------------------------------------------------
# E. Description structure
# ------------------------------------------------------------

DESCRIPTION_LABELS = [
    "Geography:",
    "Sex:",
    "Age class:",
    "Country/region of birth:"
]


if records_evaluable:

    description_dimension_labels_valid = all(
        isinstance(
            record,
            dict
        )
        and isinstance(
            record.get(
                "Description"
            ),
            str
        )
        and all(
            label in record.get(
                "Description",
                ""
            )
            for label
            in DESCRIPTION_LABELS
        )

        for record
        in extracted_records
    )


    observed_flag_segment_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Description"
                ),
                str
            )
            and "Statistical flag:"
            in record[
                "Description"
            ]
        )
    )


else:

    description_dimension_labels_valid = None

    observed_flag_segment_count = None


# ------------------------------------------------------------
# F. Source-location diagnostics
# ------------------------------------------------------------

SOURCE_LOCATION_PATTERN = re.compile(
    r"^Sheet \d+, cell [A-Z]+\d+$"
)


if records_evaluable:

    observed_source_locations = [
        record.get(
            "Source Location"
        )

        for record
        in extracted_records

        if isinstance(
            record,
            dict
        )
    ]


    source_location_format_valid = all(
        isinstance(
            location,
            str
        )
        and bool(
            SOURCE_LOCATION_PATTERN.fullmatch(
                location
            )
        )

        for location
        in observed_source_locations
    )


    observed_source_location_set = set(
        observed_source_locations
    )


    missing_source_locations = sorted(
        expected_source_location_set
        - observed_source_location_set
    )


    unexpected_source_locations = sorted(
        observed_source_location_set
        - expected_source_location_set
    )


    expected_source_locations_complete = (
        len(
            missing_source_locations
        )
        == 0
    )


    extracted_source_locations_unique = (
        len(
            observed_source_location_set
        )
        == len(
            observed_source_locations
        )
    )


else:

    observed_source_locations = None

    source_location_format_valid = None

    missing_source_locations = None

    unexpected_source_locations = None

    expected_source_locations_complete = None

    extracted_source_locations_unique = None


# ------------------------------------------------------------
# G. Exact duplicates
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(
                    field
                ),
                ensure_ascii=False,
                sort_keys=True
            )

            for field
            in EXPECTED_FIELDS
        )

        for record
        in extracted_records

        if isinstance(
            record,
            dict
        )
    )


    duplicate_records = [
        list(key)

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count
        == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# H. Scope-check table
# ------------------------------------------------------------

scope_check_rows = []


if records_evaluable:

    observed_location_counter = Counter(
        observed_source_locations
    )


    for expected_location in (
        expected_source_locations
    ):

        occurrence_count = (
            observed_location_counter.get(
                expected_location,
                0
            )
        )


        scope_check_rows.append(
            {
                "Expected Source Location":
                    expected_location,

                "Observed Count":
                    occurrence_count,

                "Valid":
                    occurrence_count == 1
            }
        )


scope_check_df = pd.DataFrame(
    scope_check_rows
)


if records_evaluable:

    fixed_scope_valid = (
        not scope_check_df.empty
        and bool(
            scope_check_df[
                "Valid"
            ].all()
        )
        and not unexpected_source_locations
    )


else:

    fixed_scope_valid = None


scope_check_df.to_csv(
    SCOPE_CHECK_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# I. Value-type diagnostics
# ------------------------------------------------------------

if records_evaluable:

    numeric_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Value"
                ),
                (int, float)
            )
            and not isinstance(
                record.get(
                    "Value"
                ),
                bool
            )
        )
    )


    null_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Value"
            )
            is None
        )
    )


else:

    numeric_value_count = None

    null_value_count = None


# ------------------------------------------------------------
# J. Content/scope diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "year_counts_match_reference":
        year_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "category_constant_valid":
        category_constant_valid,

    "topic_constant_valid":
        topic_constant_valid,

    "unit_constant_valid":
        unit_constant_valid,

    "constant_fields_valid":
        constant_fields_valid,

    "description_dimension_labels_valid":
        description_dimension_labels_valid,

    "observed_flag_segment_count":
        observed_flag_segment_count,

    "source_location_format_valid":
        source_location_format_valid,

    "expected_source_locations_complete":
        expected_source_locations_complete,

    "source_locations_unique":
        extracted_source_locations_unique,

    "missing_source_locations":
        missing_source_locations,

    "unexpected_source_locations":
        unexpected_source_locations,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "fixed_scope_valid":
        fixed_scope_valid,

    "numeric_value_count":
        numeric_value_count,

    "null_value_count":
        null_value_count
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Observed record count:",
    observed_record_count
)

print(
    "Record count matches:",
    record_count_valid
)

print(
    "Category counts match:",
    category_counts_valid
)

print(
    "Year counts match:",
    year_counts_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

print(
    "Description labels valid:",
    description_dimension_labels_valid
)

print(
    "Observed flag segments:",
    observed_flag_segment_count
)

print(
    "Source-location format valid:",
    source_location_format_valid
)

print(
    "Expected source locations complete:",
    expected_source_locations_complete
)

print(
    "Source locations unique:",
    extracted_source_locations_unique
)

print(
    "Duplicate records:",
    duplicate_record_count
)

print(
    "Fixed scope valid:",
    fixed_scope_valid
)


display(
    scope_check_df.head(20)
)

In [ ]:
# ============================================================
# 8. Technical diagnostic summary and experiment metadata
# ============================================================

# ------------------------------------------------------------
# Technical/schema validity ONLY
# ------------------------------------------------------------

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(
            valid_json
        ),

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])

# ------------------------------------------------------------
# Structure-check artifact
# ------------------------------------------------------------

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "expected_year_counts":
        EXPECTED_YEAR_COUNTS,

    "observed_year_counts":
        observed_year_counts,

    "year_counts_valid":
        year_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_worksheet_count":
            16,

        "observed_worksheet_count":
            len(
                sheet_names
            ),

        "worksheet_count_verified":
            worksheet_count_valid,

        "worksheet_names_verified":
            worksheet_names_valid,

        "summary_sheet_present":
            summary_sheet_present,

        "selected_sheets_present":
            selected_sheets_present,

        "summary_metadata_valid":
            summary_metadata_valid,

        "machine_readable":
            True,

        "ocr_required":
            False
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_workbook_loading_applied":
        True,

    "diagnostic_worksheet_inspection_applied":
        True,

    "diagnostic_cell_coordinate_inspection_applied":
        True,

    "diagnostic_scope_derivation_applied":
        True,

    "diagnostically_inspected_cells_used_as_model_input":
        False,

    "derived_representation_used_as_model_input":
        False,

    "xlsx_to_text_conversion_applied":
        False,

    "xlsx_to_csv_conversion_applied":
        False,

    "worksheet_extraction_applied_to_model_input":
        False,

    "worksheet_filtering_applied_to_model_input":
        False,

    "row_filtering_applied_to_model_input":
        False,

    "column_filtering_applied_to_model_input":
        False,

    "column_restructuring_applied":
        False,

    "row_reordering_applied":
        False,

    "aggregation_applied":
        False,

    "interpolation_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "statistical_flag_reconstruction_applied":
        False,

    "value_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_xlsx_supplied":
        True,

    "selected_sheets":
        SELECTED_SHEETS,

    "selected_geographies":
        SELECTED_GEOGRAPHIES,

    "selected_years":
        SELECTED_YEARS,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_year_counts":
            EXPECTED_YEAR_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_values_disclosed_to_model":
        False,

    "reference_record_count_explicitly_disclosed_to_model":
        False,

    "reference_year_counts_explicitly_disclosed_to_model":
        False,

    "worksheet_dimension_answers_disclosed_to_model":
        False,

    "statistical_flag_answers_disclosed_to_model":
        False,

    "source_cell_reference_answers_disclosed_to_model":
        False,

    "selected_scope_disclosed_to_model":
        True,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "selection_scope_file":
        SELECTION_SCOPE_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "observed_year_counts":
        observed_year_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "scope_check_file":
        SCOPE_CHECK_PATH.name,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "notes":
        (
            "Branch A submits the complete original D13 XLSX workbook "
            "directly to the model. OpenPyXL workbook inspection and "
            "cell-coordinate derivation are used only for source "
            "integrity and scope diagnostics and are not supplied as "
            "an alternative model representation. No workbook-to-text "
            "conversion, worksheet extraction, row or column filtering, "
            "structural conversion, normalisation, value conversion, "
            "statistical-flag reconstruction or manual correction is "
            "applied before extraction. The selected worksheets, "
            "geographies and years define the extraction scope and are "
            "supplied to the model, while Stage 1 numerical values, "
            "worksheet-level dimension answers, statistical-flag "
            "answers, exact reference source-cell set and explicit "
            "reference record/year counts are not supplied. Content-level "
            "validation is performed separately in Validation A — D13."
        )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "worksheet_count":
        len(
            sheet_names
        ),

    "selected_sheet_count":
        len(
            SELECTED_SHEETS
        ),

    "selected_geography_count":
        len(
            SELECTED_GEOGRAPHIES
        ),

    "selected_year_count":
        len(
            SELECTED_YEARS
        ),

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "expected_year_counts":
        EXPECTED_YEAR_COUNTS,

    "observed_year_counts":
        observed_year_counts,

    "year_counts_match":
        year_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "category_constant_valid":
        category_constant_valid,

    "topic_constant_valid":
        topic_constant_valid,

    "unit_constant_valid":
        unit_constant_valid,

    "description_dimension_labels_valid":
        description_dimension_labels_valid,

    "observed_flag_segment_count":
        observed_flag_segment_count,

    "source_location_format_valid":
        source_location_format_valid,

    "expected_source_locations_complete":
        expected_source_locations_complete,

    "source_locations_unique":
        extracted_source_locations_unique,

    "fixed_scope_valid":
        fixed_scope_valid,

    "numeric_value_count":
        numeric_value_count,

    "null_value_count":
        null_value_count,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes":
        (
            "This notebook performs source verification, representation "
            "characterisation, D13 Branch A direct-XLSX execution "
            "preservation, technical/schema checks and document-specific "
            "scope diagnostics only. Exact numerical, dimension, "
            "statistical-flag and source-location agreement with the "
            "fixed Stage 1 reference dataset is evaluated separately "
            "in Validation A — D13."
        )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\n" + "=" * 60
)

print(
    "D13 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed          :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Raw response preserved          :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                      :",
    valid_json
)

print(
    "Records evaluable               :",
    records_evaluable
)

print(
    "Expected records                :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records                :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches            :",
    record_count_valid
)

print(
    "Category counts match           :",
    category_counts_valid
)

print(
    "Year counts match               :",
    year_counts_valid
)

print(
    "Record schema valid             :",
    record_schema_valid
)

print(
    "Field types valid               :",
    field_types_valid
)

print(
    "Description labels valid        :",
    description_dimension_labels_valid
)

print(
    "Source locations complete       :",
    expected_source_locations_complete
)

print(
    "Fixed scope valid               :",
    fixed_scope_valid
)

print(
    "Structurally evaluable          :",
    structurally_evaluable
)

print(
    "Content validation performed   : False"
)

print(
    "Next step                       : Validation A — D13"
)


# ------------------------------------------------------------
# Required artifacts
# ------------------------------------------------------------

required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    SELECTION_SCOPE_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    SCOPE_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):

    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name
    for path
    in required_output_paths
    if not path.exists()
]


if missing_output_files:

    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D13 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )